In [ ]:
# Disable ipywidgets progress bars to avoid context issues
import os

from collections import defaultdict
import logging
import json
import argparse

from datasets import load_dataset, Dataset
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader as TorchDataLoader
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from transformers.trainer_utils import set_seed

from src.activation_capture import ActivationCapture, Hook
from src.utilities.device_utils import get_device, empty_cache

from cett import quantile, cett_from_threshold, calculate_threshold, find_thresholds

In [2]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"
device = "cuda"
dataset_name="allenai/c4"
dataset_config="en"
seed=42
max_samples=500
max_length=256
batch_size=1
num_workers=8
cett_target=0.2
n_thresholds=1000

device = get_device(device)
set_seed(seed)

In [3]:
find_thresholds(
    model_name=model_name, 
    dataset_name=dataset_name, 
    dataset_config=dataset_config,
    max_samples=max_samples, 
    cett_target=cett_target, 
    n_thresholds=n_thresholds,
    batch_size=batch_size,
    save_path="thresholds.json",
    seed=seed,
    device=device,
)

INFO:cett:Loading model: meta-llama/Llama-3.2-1B-Instruct
/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f92374d4-f470-4df0-85f2-3c41ae678bb6)')' thrown while requesting HEAD https://huggingface.co/datasets/allenai/c4/resolve/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/dataset_infos.json
Retrying in 1s [Retry 1/5].


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
INFO:cett:Beginning to compute thresholds using 500 samples
  0%|          | 0/500 [00:00<?, ?it/s]/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/datasets/formatting/torch_formatter.py:222: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(value, **{**default_dtype, **self.torch_tensor_kwargs})
/home/shadeform/sparse_tra

ReadTimeout: Caught ReadTimeout in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
  File "/usr/lib/python3.10/http/client.py", line 1375, in getresponse
    response.begin()
  File "/usr/lib/python3.10/http/client.py", line 318, in begin
    version, status, reason = self._read_status()
  File "/usr/lib/python3.10/http/client.py", line 279, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
  File "/usr/lib/python3.10/ssl.py", line 1303, in recv_into
    return self.read(nbytes, buffer)
  File "/usr/lib/p

/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/datasets/formatting/torch_formatter.py:222: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(value, **{**default_dtype, **self.torch_tensor_kwargs})
/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/datasets/formatting/torch_formatter.py:222: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(value, **{**default_dtype, **self.torch_tensor_kwargs})


In [ ]:
find_thresholds(
    model_name="meta-llama/Llama-3.2-3B-Instruct", 
    dataset_name=dataset_name, 
    dataset_config=dataset_config,
    max_samples=max_samples, 
    cett_target=cett_target, 
    n_thresholds=n_thresholds,
    save_path="thresholds.json",
    seed=seed,
    device=device,
)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
)
model = model.to(device)
model.eval()
model.activation_capture = ActivationCapture(model)
model.activation_capture.register_hooks(hooks=[Hook.UP])

/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may hav

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
if dataset_config:
    dataset = load_dataset(
        dataset_name, dataset_config, split="train", streaming=True
    )
else:
    dataset = load_dataset(dataset_name, split="train", streaming=True)
dataset = dataset.shuffle(buffer_size=10000, seed=seed)

def sample_and_tokenize(examples):
    """Sample text chunks before tokenization for efficiency using vectorized operations."""
    texts = examples["text"]
    tokenized = tokenizer(
        texts, 
        max_length=max_length,
        truncation=True,
        return_tensors="pt"
    )

    # Convert to lists
    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"]
    }

# Tokenize
dataset = Dataset.from_list(list(dataset.take(max_samples)))
dataset = dataset.shuffle(seed=seed).map(sample_and_tokenize, batched=False)
dataset = dataset.with_format("torch")

dataloader = TorchDataLoader(dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=False, prefetch_factor=2)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: d4e8d0bc-1aff-4906-bffd-25afbb824559)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/datasets/allenai/c4/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/README.md
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 764962e9-2283-4b14-b5a6-feacba70777e)')' thrown while requesting GET https://huggingface.co/api/resolve-cache/datasets/allenai/c4/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/README.md
Retrying in 1s [Retry 1/5].


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [5]:
with torch.no_grad():
    all_col_norms = {layer_idx: layer.mlp.down_proj.weight.norm(dim=0) \
                        for layer_idx, layer in enumerate(model.activation_capture.get_layers())}

In [13]:
thresholds = defaultdict(list)
with torch.no_grad():
    for batch in tqdm(dataloader, total=max_samples//batch_size):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
    
        _ = model(input_ids=input_ids.squeeze(1), attention_mask=attention_mask.squeeze(1))

        for layer_idx, layer in enumerate(model.activation_capture.get_layers()):
            down_weight = layer.mlp.down_proj.weight
            col_norms = all_col_norms[layer_idx]
            activations = model.activation_capture.mlp_activations[Hook.UP][layer_idx]
            threshold = calculate_threshold(activations, down_weight, col_norms, cett_target, n_thresholds)
            thresholds[layer_idx].append(threshold)
        
        model.activation_capture.clear_captures()
        empty_cache(device)

  0%|          | 0/500 [00:00<?, ?it/s]/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/datasets/formatting/torch_formatter.py:222: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(value, **{**default_dtype, **self.torch_tensor_kwargs})
/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/datasets/format

In [11]:
dataset

IterableDataset({
    features: Unknown,
    num_shards: 1024
})

In [12]:
dataset[0]

In [ ]:
for layer_idx, layer_thresholds in thresholds.items():
    thresholds[layer_idx] = sum(layer_thresholds) / len(layer_thresholds)

SyntaxError: invalid syntax (131496961.py, line 2)

In [15]:
thresholds

defaultdict(list,
            {0: tensor(0.0070),
             1: tensor(0.0079),
             2: tensor(0.0114),
             3: tensor(0.0139),
             4: tensor(0.0153),
             5: tensor(0.0165),
             6: tensor(0.0166),
             7: tensor(0.0180),
             8: tensor(0.0201),
             9: tensor(0.0227),
             10: tensor(0.0249),
             11: tensor(0.0263),
             12: tensor(0.0269),
             13: tensor(0.0337),
             14: tensor(0.0445),
             15: tensor(0.1110)})